In [21]:
import pandas as pd
from pprint import pprint
#filter to only the rows where the model is resnet50
# gb = cc.groupby(["backbone", "pooling", "sampler", "weight_config", "fulltune"])
coralcam_groups = ['aggression', 'biting']
fishfollow_groups = ['habitat', 'movement', 'bites', 'social_interaction', 'not_visible']
cc = pd.read_csv("../results/eccv/coralcam_eccv_label_tolerance_7.csv")
ff = pd.read_csv("../results/eccv/fishfollow_eccv_label_tolerance_7.csv")

In [22]:
ff['weight_config']

0                         {'weight_method': 'uniform'}
1    {'weight_method': 'focal_loss', 'focal_loss_al...
2                         {'weight_method': 'uniform'}
3    {'weight_method': 'focal_loss', 'focal_loss_al...
4                         {'weight_method': 'uniform'}
5    {'weight_method': 'focal_loss', 'focal_loss_al...
6                         {'weight_method': 'uniform'}
7    {'weight_method': 'focal_loss', 'focal_loss_al...
Name: weight_config, dtype: object

In [23]:
import re
def parse_weight_config(weight_config):
    s = str(weight_config)

    # Case 1: plain string (ex: weight_method='focal_loss')
    match1 = re.search(
        r"weight_method\s*[:=]\s*['\"]?([A-Za-z_]+)",
        s
    )

    # Case 2: dict-like (ex: {'weight_method': 'focal_loss', ...})
    match2 = re.search(
        r"'weight_method'\s*:\s*['\"]?([A-Za-z_]+)",
        s
    )

    if match1:
        return match1.group(1)
    if match2:
        return match2.group(1)
    return None

cc['ci_strategy'] = cc['weight_config'].apply(parse_weight_config)
ff['ci_strategy'] = ff['weight_config'].apply(parse_weight_config)

cc.fillna({"freeze_backbone": False}, inplace=True)
cc['fulltune_status'] = cc['fulltune'] & ~cc['freeze_backbone']

ff.fillna({"freeze_backbone": False}, inplace=True)
ff['fulltune_status'] = ff['fulltune'] & ~ff['freeze_backbone']

In [24]:
ff.groupby(['backbone', 'ci_strategy']).groups

{('dinov3_base', 'focal_loss'): [1, 3], ('dinov3_base', 'uniform'): [0, 2], ('videomae_large', 'focal_loss'): [5, 7], ('videomae_large', 'uniform'): [4, 6]}

In [25]:
ff.columns

Index(['id', 'model', 'epochs', 'dataset', 'monitor', 'pooling', 'sampler',
       'shuffle', 'backbone', 'fulltune', 'optimizer', 'batch_size',
       'classifier', 'label_type', 'val_subset', 'train_subset',
       'weight_decay', 'learning_rate', 'sliding_style', 'weight_config',
       'freeze_backbone', 'training_entity', 'training_run_id',
       'training_project', 'test_sliding_style', 'max_samples_per_class',
       'run_id', 'f1_micro', 'f1_macro', 'precision_micro', 'precision_macro',
       'recall_micro', 'recall_macro', 'acc', 'mAP', 'f1_per_class',
       'precision_per_class', 'recall_per_class', 'positive_per_class',
       'ap_per_class', 'confusion_matrix', 'habitat_f1_micro',
       'habitat_f1_macro', 'habitat_precision_micro',
       'habitat_precision_macro', 'habitat_recall_micro',
       'habitat_recall_macro', 'habitat_acc', 'habitat_mAP', 'bites_f1_micro',
       'bites_f1_macro', 'bites_precision_micro', 'bites_precision_macro',
       'bites_recall_micro', 

In [26]:
def get_full_table(df: pd.DataFrame, group_names, max_metric):
    df = df[df['fulltune_status'] == False]
    best_rows = df.loc[df.groupby(["backbone", 'pooling', 'ci_strategy'])[max_metric].idxmax()]
    return best_rows[["backbone", "pooling", "ci_strategy"] + [max_metric] +
            [f"{group}_{max_metric}" for group in group_names] ]

In [27]:
get_full_table(cc, coralcam_groups, "f1_macro")

,backbone,pooling,ci_strategy,f1_macro,aggression_f1_macro,biting_f1_macro
1,dinov3_base,attention,focal_loss,0.288791,0.000000,0.433186
0,dinov3_base,attention,uniform,0.309183,0.000000,0.463774
3,dinov3_base,mean,focal_loss,0.241447,0.004955,0.359694
2,dinov3_base,mean,uniform,0.025878,0.000000,0.038817
5,videomae_large,attention,focal_loss,0.450383,0.331539,0.509804
4,videomae_large,attention,uniform,0.260826,0.086207,0.348136
7,videomae_large,mean,focal_loss,0.230205,0.065014,0.312800
6,videomae_large,mean,uniform,0.243682,0.115523,0.307761


In [28]:
get_full_table(cc, coralcam_groups, "mAP")

,backbone,pooling,ci_strategy,mAP,aggression_mAP,biting_mAP
1,dinov3_base,attention,focal_loss,0.295154,0.001124,0.442170
0,dinov3_base,attention,uniform,0.276029,0.001799,0.413143
3,dinov3_base,mean,focal_loss,0.165177,0.001869,0.246830
2,dinov3_base,mean,uniform,0.078043,0.000000,0.117065
5,videomae_large,attention,focal_loss,0.201312,0.204558,0.199689
4,videomae_large,attention,uniform,0.083490,0.017200,0.116635
7,videomae_large,mean,focal_loss,0.143246,0.158596,0.135571
6,videomae_large,mean,uniform,0.126821,0.159914,0.110275


In [29]:
get_full_table(ff, fishfollow_groups, "f1_macro")

,backbone,pooling,ci_strategy,f1_macro,habitat_f1_macro,movement_f1_macro,bites_f1_macro,social_interaction_f1_macro,not_visible_f1_macro
1,dinov3_base,attention,focal_loss,0.377576,0.582361,0.464945,0.108132,0.055101,0.457178
0,dinov3_base,attention,uniform,0.320052,0.577074,0.444237,0.002041,0.013156,0.188997
3,dinov3_base,mean,focal_loss,0.367592,0.560476,0.464037,0.090770,0.135412,0.369358
2,dinov3_base,mean,uniform,0.317454,0.567811,0.441232,0.000966,0.021258,0.193152
5,videomae_large,attention,focal_loss,0.386816,0.562756,0.472224,0.148980,0.121287,0.410989
4,videomae_large,attention,uniform,0.369281,0.574246,0.465322,0.131051,0.024063,0.334086
7,videomae_large,mean,focal_loss,0.354790,0.540916,0.466721,0.111044,0.038094,0.284686
6,videomae_large,mean,uniform,0.290880,0.529521,0.407153,0.007659,0.000000,0.134138


In [30]:
get_full_table(ff, fishfollow_groups, "mAP")

,backbone,pooling,ci_strategy,mAP,habitat_mAP,movement_mAP,bites_mAP,social_interaction_mAP,not_visible_mAP
1,dinov3_base,attention,focal_loss,0.304945,0.524114,0.385562,0.004033,0.033348,0.418690
0,dinov3_base,attention,uniform,0.293474,0.499634,0.378864,0.003751,0.032421,0.378264
3,dinov3_base,mean,focal_loss,0.296692,0.492631,0.391672,0.004030,0.050978,0.357673
2,dinov3_base,mean,uniform,0.297594,0.493183,0.392812,0.004293,0.050037,0.362197
5,videomae_large,attention,focal_loss,0.307633,0.506739,0.402407,0.005599,0.061159,0.389025
4,videomae_large,attention,uniform,0.306801,0.492254,0.413673,0.004968,0.042928,0.385453
7,videomae_large,mean,focal_loss,0.317525,0.515480,0.430451,0.005399,0.048272,0.364664
6,videomae_large,mean,uniform,0.307170,0.500678,0.413778,0.005545,0.059948,0.345709


In [31]:
def get_backbone_table(df: pd.DataFrame, group_names, max_metric):
    df = df[df['fulltune_status'] == False]
    best_rows = df.loc[df.groupby("backbone")[max_metric].idxmax()]
    return best_rows[["backbone"] + ["f1_macro", "mAP"] +
            [f"{group}_f1_macro" for group in group_names] +
            [f"{group}_mAP" for group in group_names]]
    

In [32]:
get_backbone_table(cc, coralcam_groups, "f1_macro")

,backbone,f1_macro,mAP,aggression_f1_macro,biting_f1_macro,aggression_mAP,biting_mAP
0,dinov3_base,0.309183,0.276029,0.000000,0.463774,0.001799,0.413143
5,videomae_large,0.450383,0.201312,0.331539,0.509804,0.204558,0.199689


In [33]:
get_backbone_table(ff, fishfollow_groups, "f1_macro")

,backbone,f1_macro,mAP,habitat_f1_macro,movement_f1_macro,bites_f1_macro,social_interaction_f1_macro,not_visible_f1_macro,habitat_mAP,movement_mAP,bites_mAP,social_interaction_mAP,not_visible_mAP
1,dinov3_base,0.377576,0.304945,0.582361,0.464945,0.108132,0.055101,0.457178,0.524114,0.385562,0.004033,0.033348,0.418690
5,videomae_large,0.386816,0.307633,0.562756,0.472224,0.148980,0.121287,0.410989,0.506739,0.402407,0.005599,0.061159,0.389025


In [34]:
def get_binary_pivot_table(df: pd.DataFrame, pivot, pivot_values: list, max_metric): 
    df = df[df['fulltune_status'] == False]
    best_rows = df.loc[df.groupby(["backbone", pivot])[max_metric].idxmax()]
    #pivot the table to have ci_strategy as columns and backbone as rows
    pivoted = best_rows.pivot(index="backbone", columns=pivot, values=["f1_macro", "mAP", 'precision_macro', 'recall_macro'])
    assert len(pivot_values) == 2, "Only supports two pivot values for now"
    pivoted["delta_precision_macro"] = (
        pivoted[("precision_macro", pivot_values[0])]
        - pivoted[("precision_macro", pivot_values[1])]
    )
    pivoted["delta_recall_macro"] = (
        pivoted[("recall_macro", pivot_values[0])]
        - pivoted[("recall_macro", pivot_values[1])]
    )
    pivoted["delta_f1_macro"] = (
        pivoted[("f1_macro", pivot_values[0])]
        - pivoted[("f1_macro", pivot_values[1])]
    )
    #delete precision_macro and recall_macro columns
    pivoted = pivoted.drop(
        columns=[("precision_macro", pivot_values[0]), ("precision_macro", pivot_values[1]), ("recall_macro", pivot_values[0]), ("recall_macro", pivot_values[1])])
    return pivoted.reset_index()


In [35]:
get_binary_pivot_table(cc, "ci_strategy", ["focal_loss", "uniform"], "f1_macro")

backbone   f1_macro                  mAP            \
ci_strategy                 focal_loss   uniform focal_loss   uniform   
0               dinov3_base   0.288791  0.309183   0.295154  0.276029   
1            videomae_large   0.450383  0.260826   0.201312  0.083490   

            delta_precision_macro delta_recall_macro delta_f1_macro  
ci_strategy                                                          
0                       -0.021950           0.035774      -0.020392  
1                        0.225482           0.042335       0.189557

In [36]:
get_binary_pivot_table(ff, "ci_strategy", ["focal_loss", "uniform"], "f1_macro")

backbone   f1_macro                  mAP            \
ci_strategy                 focal_loss   uniform focal_loss   uniform   
0               dinov3_base   0.377576  0.320052   0.304945  0.293474   
1            videomae_large   0.386816  0.369281   0.307633  0.306801   

            delta_precision_macro delta_recall_macro delta_f1_macro  
ci_strategy                                                          
0                       -0.041934           0.186035       0.057523  
1                       -0.034557           0.160629       0.017535

In [37]:
get_binary_pivot_table(cc, "pooling", ['attention', 'mean'], "f1_macro")

backbone  f1_macro                 mAP            \
pooling                 attention      mean attention      mean   
0           dinov3_base  0.309183  0.241447  0.276029  0.165177   
1        videomae_large  0.450383  0.243682  0.201312  0.126821   

        delta_precision_macro delta_recall_macro delta_f1_macro  
pooling                                                          
0                    0.060198          -0.126344       0.067735  
1                    0.221248           0.053215       0.206701

In [38]:
get_binary_pivot_table(ff, "pooling", ['attention', 'mean'], "f1_macro")

backbone  f1_macro                 mAP            \
pooling                 attention      mean attention      mean   
0           dinov3_base  0.377576  0.367592  0.304945  0.296692   
1        videomae_large  0.386816  0.354790  0.307633  0.317525   

        delta_precision_macro delta_recall_macro delta_f1_macro  
pooling                                                          
0                    0.002317           0.038157       0.009984  
1                   -0.016458           0.130756       0.032026

In [39]:
def get_resnet_table(df, group_names, max_metric):
    dino = df[df['backbone'] == 'dinov3_large']
    resnet_frozen = df[(df['backbone'] == 'resnet50') & (df['fulltune_status'] == False)]
    resnet_fulltune = df[(df['backbone'] == 'resnet50') & (df['fulltune_status'] == True)]

    dino_best = dino.loc[dino[max_metric].idxmax()]
    resnet_frozen_best = resnet_frozen.loc[resnet_frozen[max_metric].idxmax()]
    resnet_fulltune_best = resnet_fulltune.loc[resnet_fulltune[max_metric].idxmax()]
    
    
    # columns you care about
    cols = (
        ["backbone", "fulltune_status", "f1_macro", "mAP"]
        + [f"{g}_f1_macro" for g in group_names]
        + [f"{g}_mAP" for g in group_names]
    )

    # each *_best is a Series → turn into 1-row DataFrame with .to_frame().T
    table = pd.concat(
        [row[cols].to_frame().T for row in [dino_best, resnet_frozen_best, resnet_fulltune_best]],
        ignore_index=True,
    )

    return table

In [40]:
# get_resnet_table(cc, coralcam_groups, "f1_macro")

In [41]:
# get_resnet_table(ff, fishfollow_groups, "f1_macro")